# sld-poc — Stage 3: SPECTRAL latent diffusion (JAX / TPU)

Same denoiser as Stage 2, but it diffuses the latents in the **frequency domain** (orthonormal DCT across positions), so generation goes **coarse → fine** — low-frequency global structure first, high-frequency detail last.

**The deliverable:** run this `gen` and compare it against **Stage 2's** `gen` — that's your baseline-vs-spectral result.

Same robust `load()` pattern (force-pulls + clean re-import). Run on a **TPU** runtime, top to bottom. Requires Stage A's `codec.pkl` on Drive.

## 1. One-time setup
Mounts Drive, clones, installs, defines `load()`. Run once per session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
REPO_DIR = '/content/sld-poc'
DATA_DIR = '/content/drive/MyDrive/sld-poc-data'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/mccooking/sld-poc.git $REPO_DIR
!pip -q install flax optax datasets tiktoken
os.makedirs(DATA_DIR, exist_ok=True)

def load():
    """Force-pull latest code and re-import spectral cleanly (clears spectral/diffusion/codec)."""
    import sys, subprocess
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', '-q'], check=False)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main', '-q'], check=False)
    src = REPO_DIR + '/src'
    if src not in sys.path: sys.path.insert(0, src)
    for m in ('spectral', 'diffusion', 'codec'): sys.modules.pop(m, None)
    import spectral
    return spectral

print('setup done | codec.pkl present:', os.path.exists(DATA_DIR + '/codec.pkl'))

## 2. Check the TPU

In [ ]:
import jax
print(jax.devices())
print('backend:', jax.default_backend())

## 3. Build spectral latents
Encodes chunks, applies the DCT across positions, and per-frequency standardizes. Prints a shape like `(200000, 16, 64)`.

In [ ]:
D = load()
P = D.paths(DATA_DIR)
import os
if os.path.exists(P['latents']): os.remove(P['latents'])   # clean rebuild
D.build_latents(P)

## 4. Train the spectral denoiser  ⟵  re-run anytime to resume
Watch `spec step ... mse ...`. Separate checkpoints from Stage 2, so the two models coexist on Drive.

In [ ]:
D = load()
D.train_diff(D.paths(DATA_DIR))

## 5. Generate (spectral) + speed
**Compare these samples to Stage 2's `gen` output.** If spectral has better global coherence, that's the headline result; if not, you've got an honest ablation either way.

In [ ]:
D = load()
D.gen(D.paths(DATA_DIR))